# Musixmatch: fill lyrics
Replace `API_KEY` or set env var `MUSIXMATCH_API_KEY`.


**Author:** Li You  

This code file use the MUSIXMATCH_API to fills the NA in Lyrics DA. AI tools were used only for revision and improvement of code quality.

In [ ]:
!pip -q install requests pandas tqdm

In [ ]:
import os, time, random
import requests
import pandas as pd
from tqdm import tqdm
from datetime import datetime

INPUT_CSV = r"C:\Users\16172\OneDrive\桌面\RA\Music&Finance\lyrics_batches_covered_Li\covered_Li_with_lyrics_progress.csv"
BACKUP_DIR = os.path.join(os.path.dirname(INPUT_CSV), "_mxm_backups")
os.makedirs(BACKUP_DIR, exist_ok=True)

API_KEY = os.environ.get("MUSIXMATCH_API_KEY", "").strip() or "YOUR_API_KEY"
BASE_URL = "https://api.musixmatch.com/ws/1.1"

SLEEP_BETWEEN_CALLS = 0.35
MAX_RETRIES = 4
TIMEOUT = 25
USER_AGENT = "Mozilla/5.0 (lyrics-research)"

if not API_KEY or API_KEY == "YOUR_API_KEY":
    raise ValueError("Set MUSIXMATCH_API_KEY env var, or replace API_KEY in this cell.")

In [ ]:
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

# drop Unnamed:* junk columns
unnamed = [c for c in df.columns if c.strip().lower().startswith("unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

for col in ["lyrics_status", "lyrics", "mxm_track_id", "lyrics_error"]:
    if col not in df.columns:
        df[col] = ""

df.head(3)

In [ ]:
# detect track/artist columns (edit here if your column names differ)
def pick_col(cands, cols):
    m = {c.lower(): c for c in cols}
    for x in cands:
        if x.lower() in m:
            return m[x.lower()]
    return None

title_col = pick_col(["track_name","title","song","song_name","track","name"], df.columns)
artist_col = pick_col(["artist_name","artist","singer","performer"], df.columns)
if not title_col or not artist_col:
    raise ValueError(f"Cannot detect title/artist columns. title_col={title_col}, artist_col={artist_col}. Please hardcode them here.")

title_col, artist_col

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

def mxm_get(endpoint, params, max_retries=MAX_RETRIES):
    url = f"{BASE_URL}/{endpoint}"
    params = dict(params)
    params["apikey"] = API_KEY
    last_err = ""

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, params=params, timeout=TIMEOUT)
            data = r.json() if r.text else None
        except Exception as e:
            last_err = f"request_exception:{repr(e)}"
            time.sleep(SLEEP_BETWEEN_CALLS + random.random()*0.3)
            continue

        if r.status_code == 429:
            wait = min(8, 0.8 * (2 ** (attempt - 1))) + random.random()*0.5
            last_err = f"http_429_wait_{wait:.2f}s"
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            wait = min(6, 0.6 * (2 ** (attempt - 1))) + random.random()*0.4
            last_err = f"http_{r.status_code}_wait_{wait:.2f}s"
            time.sleep(wait)
            continue

        mxm_status = None
        try:
            mxm_status = data["message"]["header"]["status_code"]
        except Exception:
            mxm_status = None

        if mxm_status == 200:
            return True, 200, data, ""

        last_err = f"mxm_status_{mxm_status}"
        if mxm_status in (401, 402, 403, 404):
            return False, mxm_status, data, last_err

        time.sleep(SLEEP_BETWEEN_CALLS + random.random()*0.3)

    return False, -1, None, last_err

def find_track_id(track_name, artist_name):
    ok, status, data, err = mxm_get("matcher.track.get", {"q_track": track_name, "q_artist": artist_name})
    if not ok:
        return "", f"matcher:{err}"
    try:
        track = data["message"]["body"]["track"]
        tid = str(track.get("track_id", "")).strip()
        return tid, ("" if tid else "matcher:no_track_id")
    except Exception as e:
        return "", f"matcher:parse:{repr(e)}"

def fetch_lyrics(track_id):
    ok, status, data, err = mxm_get("track.lyrics.get", {"track_id": track_id})
    if not ok:
        return "", f"lyrics:{err}"
    try:
        lyr = data["message"]["body"]["lyrics"].get("lyrics_body", "") or ""
        return lyr, ("" if lyr.strip() else "lyrics:empty")
    except Exception as e:
        return "", f"lyrics:parse:{repr(e)}"

In [ ]:
mask = df["lyrics_status"].str.strip().str.lower().ne("ok")
to_fix = df.index[mask].tolist()
len(to_fix)

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = os.path.join(BACKUP_DIR, f"backup_{ts}.csv")
df.to_csv(backup_path, index=False, encoding="utf-8-sig")
backup_path

In [ ]:
SAVE_EVERY = 25
processed = 0

def already_ok(i):
    return df.at[i, "lyrics_status"].strip().lower() == "ok" and (df.at[i, "lyrics"].strip() != "")

for i in tqdm(to_fix):
    if already_ok(i):
        continue

    track_name = (df.at[i, title_col] or "").strip()
    artist_name = (df.at[i, artist_col] or "").strip()
    if not track_name or not artist_name:
        df.at[i, "lyrics_status"] = "fail"
        df.at[i, "lyrics_error"] = "missing_track_or_artist"
        continue

    tid = (df.at[i, "mxm_track_id"] or "").strip()
    if not tid:
        tid, e1 = find_track_id(track_name, artist_name)
        if e1:
            df.at[i, "lyrics_status"] = "fail"
            df.at[i, "lyrics_error"] = e1
            time.sleep(SLEEP_BETWEEN_CALLS)
            continue
        df.at[i, "mxm_track_id"] = tid

    lyr, e2 = fetch_lyrics(tid)
    if e2:
        df.at[i, "lyrics_status"] = "fail"
        df.at[i, "lyrics_error"] = e2
    else:
        df.at[i, "lyrics"] = lyr
        df.at[i, "lyrics_status"] = "ok"
        df.at[i, "lyrics_error"] = ""

    processed += 1
    time.sleep(SLEEP_BETWEEN_CALLS)

    if processed % SAVE_EVERY == 0:
        df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")

df.to_csv(INPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", INPUT_CSV)

In [ ]:
df["lyrics_status"].value_counts(dropna=False).head(20)